# Stage I. Baselines and the equal-load comparison (Table *tab:stage1*)

The baselines are (i) SwinV2-L at its native 384 x 384, (ii) a single ConvNeXt V2-L at 512 x 512
and (iii) five ConvNeXt V2-L fold models at 512 x 512, all with the letterbox front-end of the
earlier pipeline (no green-channel boost, coarse dropout, elastic deformation and mixup still on).
To remove the effect of different operating points, every model is thresholded to route the same
number of test images to Stage II: the 19,153 images that the 768-pixel gate passes at the
rounded threshold 0.2392.

**Steps.**
1. Compare the training presets.
2. Train the baselines (optional; long).
3. Score the test pool with each baseline (optional; produces the cached predictions).
4. Seven-model hybrid at the out-of-fold threshold.
5. Equal-load comparison, Spearman correlation and ensembles; decision.
6. Table *tab:stage1*.

## 0. Configuration

In [ ]:
from pathlib import Path

DATA_ROOT = Path("data")
TRAIN_CSV = DATA_ROOT / "train" / "train_grade.csv"
TRAIN_IMG_DIR = DATA_ROOT / "train" / "train_image"
TEST_CSV = DATA_ROOT / "test" / "test_grade.csv"
TEST_IMG_DIR = DATA_ROOT / "test" / "test_image"

OUT_ROOT = Path("outputs") / "stage1"
BASE_DIR = OUT_ROOT / "baselines"
CKPTS = {                                   # (preset, checkpoint) of each cached model
    "fold_1": ("convnextv2_512_fold", BASE_DIR / "convnextv2_512_folds" / "fold_1_best.pth"),
    "fold_2": ("convnextv2_512_fold", BASE_DIR / "convnextv2_512_folds" / "fold_2_best.pth"),
    "fold_3": ("convnextv2_512_fold", BASE_DIR / "convnextv2_512_folds" / "fold_3_best.pth"),
    "fold_4": ("convnextv2_512_fold", BASE_DIR / "convnextv2_512_folds" / "fold_4_best.pth"),
    "fold_5": ("convnextv2_512_fold", BASE_DIR / "convnextv2_512_folds" / "fold_5_best.pth"),
    "swinv2": ("swinv2_384", BASE_DIR / "swinv2_384" / "best_auc_model.pth"),
    "single_cnx": ("convnextv2_512", BASE_DIR / "convnextv2_512" / "best_auc_model.pth"),
}
OOF_CSV = BASE_DIR / "convnextv2_512_folds" / "oof_predictions.csv"
CACHE_DIR = BASE_DIR / "hybrid_cache"                 # test_preds_<name>.csv (stem, prob)
CACHE_768 = OUT_ROOT / "eval" / "test_preds_768.csv"  # notebook 02
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_BASELINES = False     # step 2: retrains all seven baseline models
PREDICT_BASELINES = False   # step 3: rescoring the test pool with all seven models
NUM_WORKERS = 0

import dataclasses
import numpy as np
import pandas as pd
from hierarchiretina.stage1 import data, engine, losses
from hierarchiretina.stage1 import evaluate as ev
from hierarchiretina.stage1 import model as M
from hierarchiretina.stage1.config import EQUAL_LOAD_REFERENCE_THRESHOLD, PRESETS, get_preset
from hierarchiretina.stage1.preprocessing import get_preprocess

DEVICE = engine.get_device()

## 1. Training presets

Values copied from the notebooks that produced each checkpoint. Note the differences from the
768-pixel gate besides resolution: letterbox front-end, stronger augmentation with coarse
dropout and elastic deformation, mixup, no Grade-1 over-sampling, and other optimiser settings.

In [ ]:
pd.DataFrame({k: dataclasses.asdict(v) for k, v in PRESETS.items()})

## 2. Train the baselines (optional)

* SwinV2-L 384: three-way split of the development pool (80/10/10, stratified on the binary
  label); the internal 10 % test part is not used here.
* ConvNeXt V2-L 512 (single): 85/15 split stratified on the binary label.
* ConvNeXt V2-L 512 (folds): StratifiedKFold(5, shuffle, seed 42); each fold model then predicts
  its held-out fold with 4-view TTA, giving the out-of-fold table used to calibrate the hybrid.

In [ ]:
if TRAIN_BASELINES:
    df = data.load_development_pool(TRAIN_CSV, TRAIN_IMG_DIR)
    for preset, sub in (("swinv2_384", "swinv2_384"), ("convnextv2_512", "convnextv2_512")):
        cfg = get_preset(preset)
        tr, va = data.split_development_pool(df, cfg)[:2]
        engine.seed_everything(cfg.seed, cfg.deterministic)
        tl, vl = data.build_loaders(cfg, tr, va, num_workers=NUM_WORKERS)
        model = M.build_model(cfg, pretrained=True)
        engine.fit(cfg, model, tl, vl, losses.build_criterion(cfg, tr.binary_label.values),
                   BASE_DIR / sub, DEVICE)
        del model
    engine.fit_kfold(get_preset("convnextv2_512_fold"), df, BASE_DIR / "convnextv2_512_folds",
                     DEVICE, num_workers=NUM_WORKERS)

## 3. Score the test pool with each baseline (optional)

Each model runs at its own resolution with the letterbox front-end, EMA weights and five TTA
views (identity, two flips, a random rotation within +-10 degrees and a random brightness/contrast
change of +-0.1). The SwinV2 checkpoint is loaded into the same test-time module as the ConvNeXt
models, which pools SwinV2's [B, H, W, C] features with GeM at a fixed p = 3 (the learned p is
used only during its training); this is how the cached SwinV2 predictions were produced.

In [ ]:
HYBRID_LOADER = {"swinv2": dict(seq_gem_learnable=False, input_size=None)}
if PREDICT_BASELINES:
    test_paths = data.list_images(TEST_IMG_DIR)
    keys = [p.stem for p in test_paths]
    tta5 = data.build_tta_transforms(rotate_limit=10, brightness=True)
    for name, (preset, ckpt) in CKPTS.items():
        cfg = get_preset(preset)
        model = M.build_model(cfg, pretrained=False, **HYBRID_LOADER.get(name, {}))
        M.load_eval_weights(model, ckpt)
        model.to(DEVICE)
        pre = get_preprocess(cfg.preprocess)
        predict = lambda ps: ev.tta_predict_paths(model, ps, pre, cfg.img_size, tta5, DEVICE,
                                                  cfg.amp, cfg.channels_last)
        ev.run_chunked_tta(predict, tta5, test_paths, keys, CACHE_DIR / f"test_preds_{name}.csv",
                           chunk_size=1000, seed=42, signature=ev.checkpoint_signature(ckpt))
        del model

Merge the cached predictions of the seven baselines and the 768-pixel gate with the ground
truth (58,689 rows expected).

In [ ]:
gt = data.load_test_labels(TEST_CSV)
merged = gt.copy()
for name in CKPTS:
    merged = merged.merge(ev.load_prediction_cache(CACHE_DIR / f"test_preds_{name}.csv", name),
                          on="stem", how="inner")
merged = merged.merge(ev.load_prediction_cache(CACHE_768, "cnx768"), on="stem", how="inner")
MODEL_COLS = list(CKPTS) + ["cnx768"]
print(f"{len(merged):,} images matched across {len(MODEL_COLS)} models")

## 4. Seven-model hybrid at the out-of-fold threshold

The threshold is chosen on the out-of-fold predictions of the five fold models with the rule of
the 512-pixel pipeline (2 TPR - FPR, clipped to [0.30, 0.60]); it is 0.30. Expected for the
full hybrid (variant C): AUC 0.9093, sensitivity 0.8140, specificity 0.8784.

In [ ]:
oof = pd.read_csv(OOF_CSV)
oof_thr = ev.find_optimal_threshold(oof.true_label, oof.oof_prob, "2tpr-fpr", (0.30, 0.60))
print(f"OOF AUC {ev.compute_metrics(oof.true_label, oof.oof_prob)['auc']:.4f} | "
      f"OOF threshold {oof_thr:.4f}")
ev.hybrid_variants(merged, oof_thr).round(4)

## 5. Equal-load comparison and ensemble decision

Every model and ensemble is thresholded to emit exactly as many positives as the 768-pixel gate
at 0.2392 (19,153). Ensembles average either probabilities or per-model percentile ranks. The
decision rule was fixed before looking at the results: keep an ensemble only if it gains at least
+0.003 AUC over the 768-pixel model alone *and* does not route fewer Grade-1 images.

In [ ]:
N_POS = int((merged.cnx768 >= EQUAL_LOAD_REFERENCE_THRESHOLD).sum())
res = ev.equal_load_comparison(merged, N_POS, MODEL_COLS)
print(f"equal routing load: {N_POS:,} images")
res["singles"].round(4)

Spearman correlation of each baseline's predictions with the 768-pixel gate (paper:
0.70-0.76).

In [ ]:
res["spearman"].round(3).to_frame("spearman_vs_768")

In [ ]:
res["ensembles"].round(4)

The best ensemble (768 x 5 + all seven, probability mean) gains about +0.004 AUC, which
passes the pre-registered rule, but it has no leakage-free set on which to choose its threshold:
the 768-pixel validation split overlaps the training data of the fold, SwinV2 and single 512-pixel
models, and only the fold models have out-of-fold predictions. The single 768-pixel model with
its validation-derived threshold is therefore deployed.

In [ ]:
res["verdict"]

## 6. Table *tab:stage1*

Expected (paper): SwinV2-L 0.884 / 0.777 / 0.855 / 0.458 / 3,769; fold models 0.889-0.902;
ConvNeXt V2-L 512 0.904 / 0.810 / 0.869 / 0.528 / 3,214; hybrid 0.909 / 0.818 / 0.872 / 0.534 /
3,079; ours 0.929 / 0.854 / 0.887 / 0.645 / 2,459; + hybrid 5x weight 0.933 / 0.862 / 0.890 /
0.643 / 2,336.

In [ ]:
s = res["singles"].set_index("variant")
e = res["ensembles"].set_index("variant")
folds = s.loc[ev.FOLD_COLS]
COLS = {"auc": ".3f", "sens": ".3f", "spec": ".3f", "g1_acc": ".3f", "fn": ",.0f"}


def row(model, size, r):
    return [model, size] + [f"{r[c]:{f}}" for c, f in COLS.items()]


table = pd.DataFrame([
    row("SwinV2-L", 384, s.loc["swinv2"]),
    ["ConvNeXt V2-L (fold models)", 512] + [f"{folds[c].min():{f}}-{folds[c].max():{f}}"
                                            for c, f in COLS.items()],
    row("ConvNeXt V2-L", 512, s.loc["single_cnx"]),
    row("Hybrid ensemble (7 models)", "mixed", e.loc["old hybrid (7, no 768)  [prob]"]),
    row("ConvNeXt V2-L (ours)", 768, s.loc["cnx768"]),
    row("  + hybrid, 5x weight", "mixed", e.loc["768x5 + all 7  [prob]"]),
], columns=["model", "input", "AUC", "sens", "spec", "G1 routed", "FN"])
table.to_csv(OUT_ROOT / "eval" / "table_stage1.csv", index=False)
res["ensembles"].to_csv(OUT_ROOT / "eval" / "ensemble_decision.csv", index=False)
table